In [ ]:
#Cargamos la API KEY y descargamos el inventario de estaciones. 

import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("AEMET_API_KEY")
BASE = "https://opendata.aemet.es/opendata/api"

r = requests.get(
    f"{BASE}/valores/climatologicos/inventarioestaciones/todasestaciones",
    params={"api_key": API_KEY},
    timeout=30,
)
print(r.status_code, r.json().get("descripcion"))

d = requests.get(r.json()["datos"], timeout=30)
d.encoding = "latin-1"
estaciones = pd.DataFrame(d.json())

print(estaciones.shape)
estaciones.head()

200 exito
(926, 7)


,latitud,provincia,altitud,indicativo,nombre,indsinop,longitud
0,394924N,ILLES BALEARS,490,B013X,"ESCORCA, LLUC",08304,025309E
1,394744N,BALEARES,5,B051A,"SÓLLER, PUERTO",08316,024129E
2,394121N,BALEARES,60,B087X,BANYALBUFAR,,023046E
3,393446N,BALEARES,52,B103B,ANDRATX - SANT ELM,,022208E
4,393305N,BALEARES,50,B158X,"CALVIÀ, ES CAPDELLÀ",,022759E


In [2]:
#Guardamos el inventario en crudo, convertimos las coordenadas a decimal y filtramos las provincias de la cuenca para elegir estaciones.

estaciones.to_csv("../data/raw/aemet_estaciones.csv", index=False)

def a_decimal(coord):
    grados, minutos, segundos = int(coord[:-5]), int(coord[-5:-3]), int(coord[-3:-1])
    valor = grados + minutos / 60 + segundos / 3600
    return -valor if coord[-1] in "WS" else valor

estaciones["lat"] = estaciones["latitud"].apply(a_decimal)
estaciones["lon"] = estaciones["longitud"].apply(a_decimal)
estaciones["altitud"] = estaciones["altitud"].astype(int)

provincias = ["JAEN", "GRANADA", "CORDOBA", "SEVILLA", "HUELVA"]
cuenca_est = estaciones[estaciones["provincia"].str.upper().isin(provincias)]

print(cuenca_est["provincia"].value_counts())
cuenca_est[["indicativo", "nombre", "provincia", "altitud", "lat", "lon"]].sort_values(["provincia", "altitud"], ascending=[True, False])

provincia
GRANADA    21
SEVILLA    16
CORDOBA    15
HUELVA     15
JAEN       14
Name: count, dtype: int64


,indicativo,nombre,provincia,altitud,lat,lon
578,5390Y,VILLANUEVA DE CÓRDOBA,CORDOBA,750,38.333056,-4.608889
576,5346X,CARDEÑA,CORDOBA,720,38.186667,-4.272500
583,5427X,DOÑA MENCÍA,CORDOBA,650,37.556111,-4.353889
514,4263X,VALSEQUILLO,CORDOBA,575,38.409444,-5.350556
515,4267X,HINOJOSA DEL DUQUE,CORDOBA,551,38.498611,-5.121389
...,...,...,...,...,...,...
610,5788X,"TOMARES, ZAUDÍN",SEVILLA,84,37.361667,-6.059722
613,5835X,CARRIÓN DE LOS CÉSPEDES,SEVILLA,76,37.360000,-6.334167
604,5702X,CARMONA,SEVILLA,50,37.565556,-5.740556
609,5783,SEVILLA AEROPUERTO,SEVILLA,34,37.416667,-5.879167


In [3]:
#Creamos una funcion reutilizable para pedir datos a AEMET y comprobamos qué estaciones tienen datos al principio (1994) y al final (2026)
#del periodo.

import time

DIARIOS = "/valores/climatologicos/diarios/datos/fechaini/{ini}T00:00:00UTC/fechafin/{fin}T23:59:59UTC/estacion/{idema}"

def pedir_aemet(endpoint, reintentos=3):
    for _ in range(reintentos):
        r = requests.get(f"{BASE}{endpoint}", params={"api_key": API_KEY}, timeout=30)
        if r.status_code == 429:
            time.sleep(60)
            continue
        meta = r.json()
        if meta.get("estado") != 200:
            return None
        d = requests.get(meta["datos"], timeout=60)
        d.encoding = "latin-1"
        return d.json()
    return None

filas = []
for idema in cuenca_est["indicativo"]:
    ini = pedir_aemet(DIARIOS.format(ini="1994-02-01", fin="1994-07-31", idema=idema))
    time.sleep(1.5)
    fin = pedir_aemet(DIARIOS.format(ini="2026-04-01", fin="2026-09-30", idema=idema))
    time.sleep(1.5)
    filas.append({"indicativo": idema, "dias_1994": len(ini or []), "dias_2026": len(fin or [])})

disp = pd.DataFrame(filas).merge(cuenca_est[["indicativo", "nombre", "provincia", "altitud"]], on="indicativo")
disp.to_csv("../data/raw/aemet_disponibilidad.csv", index=False)
disp.sort_values(["dias_1994", "dias_2026"], ascending=False).head(30)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [4]:
def pedir_aemet(endpoint, reintentos=4):
    for intento in range(reintentos):
        try:
            r = requests.get(f"{BASE}{endpoint}", params={"api_key": API_KEY}, timeout=30)
            if r.status_code == 429:
                time.sleep(60)
                continue
            meta = r.json()
            if meta.get("estado") == 404:
                return []
            if meta.get("estado") != 200:
                time.sleep(10)
                continue
            d = requests.get(meta["datos"], timeout=60)
            d.encoding = "latin-1"
            return d.json()
        except (requests.RequestException, ValueError):
            time.sleep(10 * (intento + 1))
    return None

In [5]:
if "filas" not in globals():
    filas = []
hechos = {f["indicativo"] for f in filas}

for idema in cuenca_est["indicativo"]:
    if idema in hechos:
        continue
    ini = pedir_aemet(DIARIOS.format(ini="1994-02-01", fin="1994-07-31", idema=idema))
    time.sleep(1.5)
    fin = pedir_aemet(DIARIOS.format(ini="2026-04-01", fin="2026-09-30", idema=idema))
    time.sleep(1.5)
    if ini is None or fin is None:
        print("Fallo, se reintentará:", idema)
        continue
    filas.append({"indicativo": idema, "dias_1994": len(ini), "dias_2026": len(fin)})

print("Procesadas:", len(filas), "de", len(cuenca_est))
disp = pd.DataFrame(filas).merge(cuenca_est[["indicativo", "nombre", "provincia", "altitud"]], on="indicativo")
disp.to_csv("../data/raw/aemet_disponibilidad.csv", index=False)
disp.sort_values(["dias_1994", "dias_2026"], ascending=False).head(30)

Procesadas: 81 de 81


,indicativo,dias_1994,dias_2026,nombre,provincia,altitud
13,4642E,181,183,"HUELVA, RONDA ESTE",HUELVA,18
28,5270B,181,183,JAÉN,JAEN,580
36,5402,181,183,CÓRDOBA AEROPUERTO,CORDOBA,90
44,5514,181,183,GRANADA BASE AÉREA,GRANADA,687
49,5530E,181,183,GRANADA AEROPUERTO,GRANADA,560
64,5783,181,183,SEVILLA AEROPUERTO,SEVILLA,34
67,5796,181,183,MORÓN DE LA FRONTERA,SEVILLA,87
24,5192,181,176,VILLARRODRIGO,JAEN,862
1,4267X,0,183,HINOJOSA DEL DUQUE,CORDOBA,551
2,4527X,0,183,AROCHE,HUELVA,267


In [6]:
#Descargamos los datos diarios de las 7 estaciones elegidas por tramos de 6 meses y guardamos un CSV por estación, para tener la lluvia y la
#temperatura desde 1993 hasta hoy.

from pathlib import Path

ESTACIONES = ["5270B", "5402", "5514", "5530E", "5783", "5796", "5192"]
DESDE, HASTA = "1993-07-01", "2026-10-20"

carpeta = Path("../data/raw/aemet")
carpeta.mkdir(exist_ok=True)
tramos = pd.date_range(DESDE, HASTA, freq="6MS")

for idema in ESTACIONES:
    fichero = carpeta / f"{idema}.csv"
    if fichero.exists():
        continue
    datos, fallo = [], False
    for inicio in tramos:
        fin = min(inicio + pd.DateOffset(months=6) - pd.Timedelta(days=1), pd.Timestamp(HASTA))
        res = pedir_aemet(DIARIOS.format(ini=inicio.date(), fin=fin.date(), idema=idema))
        time.sleep(1.5)
        if res is None:
            fallo = True
            break
        datos.extend(res)
    if fallo:
        print("Fallo en", idema, "- vuelve a ejecutar la celda")
        continue
    pd.DataFrame(datos).to_csv(fichero, index=False)
    print(idema, len(datos), "días")

5270B 12165 días
5402 12165 días
5514 12158 días
5530E 12165 días
5783 12165 días
5796 12165 días
5192 6294 días
